In [ ]:
import os
import torch
from kokoro import KPipeline
import soundfile as sf
from huggingface_hub import HfFileSystem, snapshot_download
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
# # Kokoro TTS Reader Example

# # 1. Quick check to ensure your NVIDIA GPU is active
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# print(f"Using device: {device}")

# # 2. Initialize the Kokoro Pipeline (using 'a' for American English)
# # The first run will automatically download the 82M model weights (~300MB)
# pipeline = KPipeline(lang_code='a', device=device)

# # 3. Define your custom training string
# practice_text = 'Hello, my name is kokoro. I can read text files out loud to you'

# # 4. Choose a native voice and speed
# # Popular voice options include: 'af_bella', 'af_sarah', 'am_adam', 'am_michael'
# voice_choice = 'af_bella' 
# speed_rate = 1.0

# print(f"Generating audio using voice: {voice_choice}...")

# # 5. Run the generator through your GPU cores
# generator = pipeline(practice_text, voice=voice_choice, speed=speed_rate)

# # Kokoro handles the text in parts; loop through and combine them
# for i, (gs, ps, audio) in enumerate(generator):
#     # Set the save output destination path
#     output_filename = f"kokoro_test_part_{i}.wav"
    
#     # Save the raw audio array directly to disk (Kokoro defaults to a clean 24kHz sample rate)
#     sf.write(output_filename, audio, 24000)
#     print(f"🎉 Success! Practice audio saved to {output_filename}")

In [ ]:
# Get list of voices from Kokoro-82M huggingface repository

# 1. Connect to the repository on Hugging Face
fs = HfFileSystem()

# 2. List all files ending in .pt inside the voices folder of the repo
# This queries the live repository, so it always sees the newest files
files = fs.glob("hexgrad/Kokoro-82M/voices/*.pt")

# 3. Extract just the filenames (e.g., "af_bella")
# files will be a list like ['hexgrad/Kokoro-82M/voices/af_bella.pt', ...]
all_available_voice_ids = [f.split('/')[-1].replace('.pt', '') for f in files]

# 4. Convert to your DataFrame
self_updating_df = pd.DataFrame(all_available_voice_ids, columns=['Voice_ID'])

# (Optional) Add your attribute parsing logic here as before
print(f"Detected {len(self_updating_df)} voices live from Hugging Face!")
self_updating_df.head(5)

In [ ]:
# Download of voices from Kokoro-82M repository

print(f"Syncing {len(self_updating_df)} voices from your dataframe to your local cache...")

# 1. Access the underlying load_voice method from your pipeline instance
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
pipeline = KPipeline(lang_code='a', device=device)
load_voice_func = pipeline.load_voice

# 2. Iterate through every Voice_ID in your dataframe column
for voice_id in tqdm(self_updating_df['Voice_ID'], desc="Caching Voices"):
    try:
        # This tells the pipeline's internal code to fetch the voice file.
        # If the file is already cached, it skips the download in milliseconds!
        load_voice_func(voice_id)
    except Exception as e:
        print(f"Could not load/download {voice_id}: {e}")

print("\n🎉 Your local hardware cache is perfectly matched with your dataframe!")

In [ ]:
# Check for voices downloaded

# 1. Construct the path to your local Hugging Face cache folder
user_home = os.path.expanduser("~")
cache_dir = os.path.join(user_home, ".cache", "huggingface", "hub", "models--hexgrad--Kokoro-82M", "snapshots")

print(f"Checking cache directory:\n{cache_dir}\n")

# 2. Check if the model folder exists yet
if os.path.exists(cache_dir):
    # Get the unique snapshot folder created by Hugging Face
    snapshots = os.listdir(cache_dir)
    if snapshots:
        # Step into the latest downloaded snapshot folder where the actual files live
        voices_path = os.path.join(cache_dir, snapshots[0], "voices")
        
        if os.path.exists(voices_path):
            downloaded_voices = [f for f in os.listdir(voices_path) if f.endswith('.pt')]
            print(f"🎉 Found {len(downloaded_voices)} downloaded voice(s):\n")
            for voice in sorted(downloaded_voices):
                # Clean up the string to show just the voice ID
                print(f" * {voice.replace('.pt', '')}")
        else:
            print("The model is cached, but no individual voice files have been pulled down yet.")
    else:
        print("Hugging Face folder exists, but no version snapshots were found.")
else:
    print("No downloaded voices found yet! Run the generator once to create the cache.")